In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============ 配置（可按需改）============
IN_XLSX = Path("btc_fit_outputs/ahr999_daily.xlsx")
IN_CSV  = Path("btc_fit_outputs/ahr999_daily.csv")
OUTDIR  = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

# 三次减半（近三轮）
HALVINGS = {
    2017: pd.Timestamp("2016-07-09"),
    2021: pd.Timestamp("2020-05-11"),
    2025: pd.Timestamp("2024-04-20"),
}

# 三次峰值（2017 修正为 12-18；2021 常用 11-10；2025 为你的假设 08-15）
PEAKS = {
    2017: pd.Timestamp("2017-12-18"),
    2021: pd.Timestamp("2021-11-10"),
    2025: pd.Timestamp("2025-08-15"),
}

PEAK_WIN_DAYS = 7   # 峰值±7天窗口内取 AHR999 最大值作为“窗口峰”
PLOT_FILENAME = OUTDIR / "ahr999_decay_fit.png"
XLSX_OUT      = OUTDIR / "ahr999_decay_analysis.xlsx"

# ============ 读取 AHR999 日线 ============
def load_ahr999(xlsx_path: Path, csv_path: Path) -> pd.DataFrame:
    if xlsx_path.exists():
        df = pd.read_excel(xlsx_path, sheet_name="AHR999")
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError("未找到 ahr999_daily.xlsx / ahr999_daily.csv，请先运行前面的 AHR999 生成脚本。")
    # 解析日期索引
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    # 关键列校验
    need_cols = {"ahr999", "price", "gma200", "estimate_price", "avg_index", "estimate_index"}
    miss = [c for c in need_cols if c not in df.columns]
    if miss:
        raise ValueError(f"输入缺少列：{miss}；请确认用的是我提供的 AHR999 导出脚本。")
    return df

ahr = load_ahr999(IN_XLSX, IN_CSV)

# ============ 按“减半→峰值”切片并抽取指标 ============
records = []
for yr in [2017, 2021, 2025]:
    halving = HALVINGS[yr]
    peak    = PEAKS[yr]
    # 为防止日期不在索引上，做对齐
    def nearest(ts):
        # 最近一天（向前优先）
        if ts in ahr.index: return ts
        before = ahr.index[ahr.index <= ts]
        after  = ahr.index[ahr.index >= ts]
        if len(before)>0: return before.max()
        return after.min()
    halving = nearest(halving)
    peak    = nearest(peak)
    seg = ahr.loc[halving:peak].copy()
    if seg.empty:
        raise ValueError(f"{yr} 区间空：{halving} → {peak}，请检查源数据覆盖。")

    ahr_at_peak = float(ahr.loc[peak, "ahr999"])

    lo, hi = peak - pd.Timedelta(days=PEAK_WIN_DAYS), peak + pd.Timedelta(days=PEAK_WIN_DAYS)
    win = ahr.loc[lo:hi, "ahr999"]
    ahr_peak_win = float(win.max())
    ahr_peak_win_date = win.idxmax()

    pre_median = float(seg["ahr999"].median())
    pre_p90    = float(seg["ahr999"].quantile(0.90))
    pre_max    = float(seg["ahr999"].max())
    days_halving_to_peak = int((peak - halving).days)

    records.append({
        "cycle_peak_year": yr,
        "halving_date": halving.date(),
        "peak_date": peak.date(),
        "days_halving_to_peak": days_halving_to_peak,
        "ahr_at_peak_date": ahr_at_peak,
        "ahr_peak_window_max": ahr_peak_win,
        "ahr_peak_window_max_date": ahr_peak_win_date.date(),
        "ahr_pre_median": pre_median,
        "ahr_pre_p90": pre_p90,
        "ahr_pre_max": pre_max,
    })

peaks_df = pd.DataFrame(records).set_index("cycle_peak_year").sort_index()
# 选用“窗口峰”作为代表值（更鲁棒）
y_obs = peaks_df["ahr_peak_window_max"].copy()

# ============ 拟合“每轮衰减”表达式 ============
# 将 2017,2021,2025 映射到 n=0,1,2
years = y_obs.index.to_list()
n = np.arange(len(years))  # 0,1,2
y = y_obs.values.astype(float)

# 1) 自由拟合（log2 线性）：log2(y) = a + b*n  → 每轮倍率 f = 2^b
log2y = np.log2(y)
A = np.vstack([np.ones_like(n), n]).T
coef, *_ = np.linalg.lstsq(A, log2y, rcond=None)
a_hat, b_hat = coef[0], coef[1]
f_hat = 2.0 ** b_hat
y_fit_free = 2.0 ** (a_hat + b_hat * n)

# 2) 三个候选固定倍率：f ∈ {1/2, 1/√2, 1/4}
candidates = {
    "half(×0.5/轮)": 0.5,
    "sqrt2(×0.707/轮)": 1/np.sqrt(2),
    "quarter(×0.25/轮)": 0.25,
}

def best_c_for_factor(f, y):
    # 令 y_n ≈ c * f^n，求最小二乘 c*
    w = f ** n
    c = (y @ w) / (w @ w)
    return c, c * w

model_rows = []
for name, f in candidates.items():
    c_star, y_fit = best_c_for_factor(f, y)
    rmse_abs = float(np.sqrt(np.mean((y - y_fit) ** 2)))
    rmse_log2 = float(np.sqrt(np.mean((np.log2(y) - np.log2(y_fit)) ** 2)))
    model_rows.append({"model": name, "factor_per_cycle": f, "c_star": c_star,
                       "rmse_abs": rmse_abs, "rmse_log2": rmse_log2})

# 自由拟合的误差（对照）
rmse_abs_free  = float(np.sqrt(np.mean((y - y_fit_free) ** 2)))
rmse_log2_free = float(np.sqrt(np.mean((np.log2(y) - np.log2(y_fit_free)) ** 2)))
model_rows.append({"model": "free_fit(2^(a+b*n))", "factor_per_cycle": f_hat, "c_star": 2**a_hat,
                   "rmse_abs": rmse_abs_free, "rmse_log2": rmse_log2_free})

models_df = pd.DataFrame(model_rows).set_index("model").sort_values("rmse_log2")

# ============ 导出 Excel ============
with pd.ExcelWriter(XLSX_OUT, engine="openpyxl") as w:
    ahr.to_excel(w, sheet_name="ahr999_daily")
    peaks_df.to_excel(w, sheet_name="cycle_peaks")
    models_df.to_excel(w, sheet_name="decay_fits")
print("已保存分析表：", XLSX_OUT.resolve())

# ============ 画图：观测 vs 拟合 ============
plt.figure(figsize=(8,4.5))
plt.plot(n, y, marker="o", linestyle="-", label="观测（窗口峰 AHR999）")
# 画自由拟合
plt.plot(n, y_fit_free, marker="o", linestyle="--", label=f"自由拟合：×{f_hat:.3f}/轮")
# 画两个最优的候选模型（按 log2 RMSE 排名前2）
top2 = models_df.drop(index=["free_fit(2^(a+b*n))"], errors="ignore").head(2)
for mdl in top2.itertuples():
    f = mdl.factor_per_cycle
    c = mdl.c_star
    y_fit = c * (f ** n)
    plt.plot(n, y_fit, marker="o", linestyle=":", label=f"{mdl.Index}（×{f:.3f}/轮）")
plt.xticks(n, years)
plt.xlabel("循环（按峰值年）"); plt.ylabel("AHR999（窗口峰）")
plt.title("AHR999 在峰值处的跨轮衰减：观测 vs 拟合")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_FILENAME, dpi=170); plt.close()
print("已保存图：", PLOT_FILENAME.resolve())

# ============ 控制台简要汇总 ============
print("\n—— 峰值窗口内 AHR999 ——")
print(peaks_df[["ahr_peak_window_max", "ahr_peak_window_max_date"]])
print("\n—— 衰减拟合（按 log2 误差排序）——")
print(models_df)
print(f"\n自由拟合每轮倍率 f_hat = {f_hat:.4f}（与 0.5 的差 {abs(f_hat-0.5):.4f}；与 1/√2 的差 {abs(f_hat-1/np.sqrt(2)):.4f}）")


已保存分析表： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_decay_analysis.xlsx


C:\Users\nickc\AppData\Local\Temp\ipykernel_18912\3459686728.py:164: UserWarning: Glyph 24490 (\N{CJK UNIFIED IDEOGRAPH-5FAA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_18912\3459686728.py:164: UserWarning: Glyph 29615 (\N{CJK UNIFIED IDEOGRAPH-73AF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_18912\3459686728.py:164: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_18912\3459686728.py:164: UserWarning: Glyph 25353 (\N{CJK UNIFIED IDEOGRAPH-6309}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_18912\3459686728.py:164: UserWarning: Glyph 23792 (\N{CJK UNIFIED IDEOGRAPH-5CF0}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_18912\3459686728.py:164: UserWarning: Glyph 20

已保存图： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_decay_fit.png

—— 峰值窗口内 AHR999 ——
                 ahr_peak_window_max ahr_peak_window_max_date
cycle_peak_year                                              
2017                       26.085440               2017-12-17
2021                        3.798769               2021-11-09
2025                        1.267364               2025-08-14

—— 衰减拟合（按 log2 误差排序）——
                     factor_per_cycle     c_star  rmse_abs  rmse_log2
model                                                                
free_fit(2^(a+b*n))          0.220420  22.719350  2.067130   0.281886
quarter(×0.25/轮)             0.250000  25.425904  1.536270   0.468862
half(×0.5/轮)                 0.500000  21.563174  5.360762   1.494757
sqrt2(×0.707/轮)              0.707107  16.803004  8.213614   1.875764

自由拟合每轮倍率 f_hat = 0.2204（与 0.5 的差 0.2796；与 1/√2 的差 0.4867）


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ========= 只改这里几行 =========
IN_XLSX = Path("btc_fit_outputs/ahr999_daily.xlsx")
IN_CSV  = Path("btc_fit_outputs/ahr999_daily.csv")
OUTDIR  = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

# 峰值与减半（2017 修正为 12-18；2021 常用 11-10；2025 为你的假设 08-15）
HALVINGS = {2017: pd.Timestamp("2016-07-09"),
            2021: pd.Timestamp("2020-05-11"),
            2025: pd.Timestamp("2024-04-20")}
PEAKS = {2017: pd.Timestamp("2017-12-18"),
         2021: pd.Timestamp("2021-11-10"),
         2025: pd.Timestamp("2025-08-15")}

# 选择你的“热度口径”（四选一）：
METRIC = "peak_month_mean"   # 可选: "peak_window_max" / "peak_month_mean" / "topk_month_mean" / "pXX_daily"
PEAK_WIN_DAYS = 7            # 用于 "peak_window_max"：峰值±N天窗口
PEAK_MONTH_PRE = 0           # 用于 "peak_month_mean"：包含峰值前多少个月
PEAK_MONTH_POST = 0          # 用于 "peak_month_mean"：包含峰值后多少个月
TOPK = 3                     # 用于 "topk_month_mean"：减半→峰值期间，取月均 AHR999 的前 K 个月均值平均
PXX = 0.95                   # 用于 "pXX_daily"：减半→峰值期间的日频 P 分位（例如 0.95）

# 对照的固定倍率
CANDIDATES = {
    "half(×0.5/轮)": 0.5,
    "sqrt2(×0.707/轮)": 1/np.sqrt(2),
    "quarter(×0.25/轮)": 0.25,
}

# ========= 读取 AHR999 日线 =========
def load_ahr999(xlsx_path: Path, csv_path: Path) -> pd.DataFrame:
    if xlsx_path.exists():
        df = pd.read_excel(xlsx_path, sheet_name="AHR999")
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError("找不到 ahr999_daily.xlsx / .csv，请先运行生成脚本。")
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    if "ahr999" not in df.columns:
        raise ValueError("输入缺少列 'ahr999'，请确认使用了我提供的导出脚本。")
    return df

ahr = load_ahr999(IN_XLSX, IN_CSV)

def nearest(ts: pd.Timestamp):
    ts = pd.Timestamp(ts)
    if ts in ahr.index: return ts
    before = ahr.index[ahr.index <= ts]
    after  = ahr.index[ahr.index >= ts]
    if len(before)>0: return before.max()
    return after.min()

# ========= 计算每轮“热度” =========
records = []
details = []  # 保存辅助信息
for yr in [2017, 2021, 2025]:
    halving = nearest(HALVINGS[yr])
    peak    = nearest(PEAKS[yr])
    seg = ahr.loc[halving:peak].copy()
    if seg.empty:
        raise ValueError(f"{yr} 区间为空：{halving}→{peak}，检查数据覆盖。")

    # 月均序列（每月平均 AHR999）
    ahr_m = seg["ahr999"].resample("MS").mean()  # 月初对齐

    if METRIC == "peak_window_max":
        lo, hi = peak - pd.Timedelta(days=PEAK_WIN_DAYS), peak + pd.Timedelta(days=PEAK_WIN_DAYS)
        val = float(ahr.loc[lo:hi, "ahr999"].max())
        note = f"窗口±{PEAK_WIN_DAYS}天内最大日值"
    elif METRIC == "peak_month_mean":
        peak_ms = pd.Timestamp(peak.year, peak.month, 1)
        months = [peak_ms + pd.DateOffset(months=i) for i in range(-PEAK_MONTH_PRE, PEAK_MONTH_POST+1)]
        vals = ahr_m.reindex(months).dropna()
        val = float(vals.mean()) if len(vals)>0 else np.nan
        note = f"峰值所在月±({PEAK_MONTH_PRE},{PEAK_MONTH_POST})个月的月均均值"
    elif METRIC == "topk_month_mean":
        vals = ahr_m.sort_values(ascending=False).head(TOPK)
        val = float(vals.mean()) if len(vals)>0 else np.nan
        note = f"减半→峰值区间内月均Top{TOPK}的平均"
    elif METRIC.startswith("p") and METRIC.endswith("_daily"):
        q = float(PXX)
        val = float(seg["ahr999"].quantile(q))
        note = f"减半→峰值区间内日频P{int(q*100)}分位"
    else:
        raise ValueError("未知 METRIC 设定")

    days_htp = int((peak - halving).days)
    records.append({"cycle_peak_year": yr, "metric_value": val,
                    "halving_date": halving.date(), "peak_date": peak.date(),
                    "days_halving_to_peak": days_htp, "metric_note": note})
    details.append({"year": yr, "monthly_mean_count": int(ahr_m.size)})

obs_df = pd.DataFrame(records).set_index("cycle_peak_year").sort_index()
y_obs = obs_df["metric_value"].astype(float).copy()
years = y_obs.index.to_list()
n = np.arange(len(years))  # 0,1,2 对应 2017,2021,2025

# ========= 拟合跨轮衰减 =========
# 自由拟合：log2(y) = a + b*n  → 每轮倍率 f_hat = 2^b，等价于 base2-decay 的 k_hat = -b
log2y = np.log2(y_obs.values)
A = np.vstack([np.ones_like(n), n]).T
coef, *_ = np.linalg.lstsq(A, log2y, rcond=None)
a_hat, b_hat = coef
f_hat = 2.0 ** b_hat
k_hat = -b_hat  # 若写成 y ≈ C / (2^k)^n，则 k = -b

y_fit_free = 2.0 ** (a_hat + b_hat * n)
rmse_abs_free  = float(np.sqrt(np.mean((y_obs.values - y_fit_free) ** 2)))
rmse_log2_free = float(np.sqrt(np.mean((log2y - np.log2(y_fit_free)) ** 2)))

# 候选固定倍率
def best_c_for_factor(f, y):  # y_n ≈ c * f^n
    w = f ** n
    c = float((y @ w) / (w @ w))
    return c, c * w

rows = []
for name, f in CANDIDATES.items():
    c_star, y_fit = best_c_for_factor(f, y_obs.values)
    rmse_abs = float(np.sqrt(np.mean((y_obs.values - y_fit) ** 2)))
    rmse_log2 = float(np.sqrt(np.mean((np.log2(y_obs.values) - np.log2(y_fit)) ** 2)))
    rows.append({"model": name, "factor_per_cycle": f, "c_star": c_star,
                 "rmse_abs": rmse_abs, "rmse_log2": rmse_log2})
rows.append({"model": "free_fit(2^(a+b*n))", "factor_per_cycle": f_hat, "c_star": 2**a_hat,
             "rmse_abs": rmse_abs_free, "rmse_log2": rmse_log2_free})
models_df = pd.DataFrame(rows).set_index("model").sort_values("rmse_log2")

# ========= 导出 =========
tag = f"{METRIC}"
xlsx_out = OUTDIR / f"ahr999_decay_{tag}.xlsx"
with pd.ExcelWriter(xlsx_out, engine="openpyxl") as w:
    ahr.to_excel(w, sheet_name="ahr999_daily")
    obs_df.to_excel(w, sheet_name="cycle_metric")
    models_df.to_excel(w, sheet_name="decay_fits")
print("已保存：", xlsx_out.resolve())

# ========= 画图 =========
plt.figure(figsize=(8,4.6))
plt.plot(n, y_obs.values, marker="o", linestyle="-", label="观测（按所选口径）")
plt.plot(n, y_fit_free, marker="o", linestyle="--", label=f"自由拟合 ×{f_hat:.3f}/轮  (k≈{k_hat:.2f})")
# 画对照里最好的两条
top2 = models_df.drop(index=["free_fit(2^(a+b*n))"], errors="ignore").head(2)
for mdl in top2.itertuples():
    f = mdl.factor_per_cycle; c = mdl.c_star
    plt.plot(n, c * (f ** n), marker="o", linestyle=":", label=f"{mdl.Index} ×{f:.3f}/轮")
plt.xticks(n, years)
plt.xlabel("循环（按峰值年）"); plt.ylabel("AHR999（所选度量）")
plt.title(f"AHR999 跨轮衰减：{METRIC}")
plt.legend(); plt.tight_layout()
plot_out = OUTDIR / f"ahr999_decay_fit_{tag}.png"
plt.savefig(plot_out, dpi=170); plt.close()
print("已保存图：", plot_out.resolve())

# 控制台快速查看
print("\n—— 每轮度量值 ——")
print(obs_df[["metric_value","metric_note","days_halving_to_peak"]])
print("\n—— 衰减拟合（按 log2 RMSE 排序）——")
print(models_df)
print(f"\n自由拟合每轮倍率 f_hat = {f_hat:.4f}，对应 base2-decay k ≈ {k_hat:.2f}（y ≈ C / (2^k)^n）")






已保存： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_decay_peak_month_mean.xlsx


C:\Users\nickc\AppData\Local\Temp\ipykernel_16388\2402538107.py:155: UserWarning: Glyph 24490 (\N{CJK UNIFIED IDEOGRAPH-5FAA}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_16388\2402538107.py:155: UserWarning: Glyph 29615 (\N{CJK UNIFIED IDEOGRAPH-73AF}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_16388\2402538107.py:155: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_16388\2402538107.py:155: UserWarning: Glyph 25353 (\N{CJK UNIFIED IDEOGRAPH-6309}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_16388\2402538107.py:155: UserWarning: Glyph 23792 (\N{CJK UNIFIED IDEOGRAPH-5CF0}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppDa

已保存图： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_decay_fit_peak_month_mean.png

—— 每轮度量值 ——
                 metric_value         metric_note  days_halving_to_peak
cycle_peak_year                                                        
2017                17.257163  峰值所在月±(0,0)个月的月均均值                   527
2021                 3.333878  峰值所在月±(0,0)个月的月均均值                   548
2025                 1.146254  峰值所在月±(0,0)个月的月均均值                   482

—— 衰减拟合（按 log2 RMSE 排序）——
                     factor_per_cycle     c_star  rmse_abs  rmse_log2
model                                                                
free_fit(2^(a+b*n))          0.257725  15.676325  1.001487   0.196021
quarter(×0.25/轮)             0.250000  17.031289  0.551175   0.213165
half(×0.5/轮)                 0.500000  14.636698  3.112213   1.175781
sqrt2(×0.707/轮)              0.707107  11.535827  5.077943   1.574597

自由拟合每轮倍率 f_hat = 0.2577，对应 base2-decay k ≈ 1.96（y ≈ C / (2^k)^n）


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ========= 配置（只改这里）=========
IN_XLSX = Path("btc_fit_outputs/ahr999_daily.xlsx")
IN_CSV  = Path("btc_fit_outputs/ahr999_daily.csv")
OUTDIR  = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

# 三轮 减半/峰值（2017 修正为 12-18；2025 为你的假设，可调整）
HALVINGS = {2017: pd.Timestamp("2016-07-09"),
            2021: pd.Timestamp("2020-05-11"),
            2025: pd.Timestamp("2024-04-20")}
PEAKS    = {2017: pd.Timestamp("2017-12-18"),
            2021: pd.Timestamp("2021-11-10"),
            2025: pd.Timestamp("2025-08-15")}

# 代表值“热度口径”集合（可增删）
METRICS = [
    ("peak_month_mean", dict(PEAK_MONTH_PRE=0, PEAK_MONTH_POST=0)),  # 峰值所在月月均
    ("topk_month_mean", dict(TOPK=3)),                                # 减半→峰值区间内月均 Top-K 的平均
    ("peak_window_max", dict(PEAK_WIN_DAYS=7)),                       # 峰值±N天日频最大
    ("pXX_daily",       dict(PXX=0.95)),                              # 日频 P 分位（如 0.95）
    ("auc_above1",      dict()),                                      # 减半→峰值期间 (ahr999-1)+ 的“面积”近似（日频求和）
    ("median_daily",    dict()),                                      # 日频中位数
]

# 固定倍率对照（每轮 ×factor）
CANDIDATES = {
    "k=0.5 (×0.707/轮)": 2**(-0.5),
    "k=1.0 (×0.5/轮)"  : 2**(-1.0),
    "k=1.5 (×0.354/轮)": 2**(-1.5),
    "k=2.0 (×0.25/轮)" : 2**(-2.0),
}

# 幂曲率扩展搜索范围（log2(y)=A+B*n^p）
P_GRID = np.round(np.linspace(0.3, 2.0, 86), 2)   # 0.3→2.0，步长≈0.02

# ========= 读取 AHR999 =========
def load_ahr999(xlsx_path: Path, csv_path: Path) -> pd.DataFrame:
    if xlsx_path.exists():
        df = pd.read_excel(xlsx_path, sheet_name="AHR999")
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError("找不到 ahr999_daily.xlsx / .csv，请先生成。")
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    if "ahr999" not in df.columns:
        raise ValueError("缺少列 'ahr999'。")
    return df

ahr = load_ahr999(IN_XLSX, IN_CSV)

def nearest(ts: pd.Timestamp):
    ts = pd.Timestamp(ts)
    if ts in ahr.index: return ts
    before = ahr.index[ahr.index <= ts]
    after  = ahr.index[ahr.index >= ts]
    if len(before)>0: return before.max()
    return after.min()

# ========= 抽取每轮代表值 =========
def cycle_segment(year):
    h, p = nearest(HALVINGS[year]), nearest(PEAKS[year])
    seg = ahr.loc[h:p].copy()
    if seg.empty: raise ValueError(f"{year} 区间为空：{h}→{p}")
    return seg, h, p

def metric_value(seg, peak_date, name, kwargs):
    if name == "peak_month_mean":
        pre  = int(kwargs.get("PEAK_MONTH_PRE", 0))
        post = int(kwargs.get("PEAK_MONTH_POST", 0))
        ms = pd.Timestamp(peak_date.year, peak_date.month, 1)
        months = [ms + pd.DateOffset(months=i) for i in range(-pre, post+1)]
        vals = seg["ahr999"].resample("MS").mean().reindex(months).dropna()
        return float(vals.mean())
    elif name == "topk_month_mean":
        k = int(kwargs.get("TOPK", 3))
        vals = seg["ahr999"].resample("MS").mean().sort_values(ascending=False).head(k)
        return float(vals.mean())
    elif name == "peak_window_max":
        d = int(kwargs.get("PEAK_WIN_DAYS", 7))
        w = ahr.loc[peak_date - pd.Timedelta(days=d) : peak_date + pd.Timedelta(days=d), "ahr999"]
        return float(w.max())
    elif name == "pXX_daily":
        q = float(kwargs.get("PXX", 0.95))
        return float(seg["ahr999"].quantile(q))
    elif name == "auc_above1":
        # (ahr999-1)+ 的日频和：热度“盈余”的粗略面积
        x = (seg["ahr999"] - 1.0).clip(lower=0.0)
        return float(x.sum())
    elif name == "median_daily":
        return float(seg["ahr999"].median())
    else:
        raise ValueError(f"未知度量：{name}")

def collect_obs(metric_name, kwargs):
    rec = []
    for yr in [2017, 2021, 2025]:
        seg, h, p = cycle_segment(yr)
        val = metric_value(seg, p, metric_name, kwargs)
        rec.append((yr, val, h.date(), p.date(), int((p-h).days)))
    df = pd.DataFrame(rec, columns=["year","metric","halving","peak","days_htp"]).set_index("year").sort_index()
    return df

# ========= 拟合与评估 =========
def rmse_log2(y, yhat):
    return float(np.sqrt(np.mean((np.log2(y) - np.log2(yhat))**2)))

def fit_free_linear(n, y):
    # log2(y) = a + b*n
    A = np.vstack([np.ones_like(n), n]).T
    coef, *_ = np.linalg.lstsq(A, np.log2(y), rcond=None)
    a, b = coef
    yhat = 2**(a + b*n)
    return dict(model="free_linear", a=a, b=b, C=2**a, k=-b, rmse_log2=rmse_log2(y, yhat), yhat=yhat)

def fit_fixed_factor(f, n, y):
    # y ≈ C * f^n → 最小二乘 C*
    w = f**n
    C = float((y @ w) / (w @ w))
    yhat = C * w
    # 对应等效 k：f = 2^{-k} → k = -log2 f
    k = -np.log2(f)
    return dict(model=f"fixed_factor_{f:.3f}", C=C, f=f, k=k, rmse_log2=rmse_log2(y, yhat), yhat=yhat)

def fit_power_curvature(n, y, p_grid=P_GRID):
    # log2(y)=A + B*(n^p)，对每个 p 线性回归 A,B，挑 RMSE_log2 最小
    best = None
    for p in p_grid:
        x = n**p
        A = np.vstack([np.ones_like(x), x]).T
        coef, *_ = np.linalg.lstsq(A, np.log2(y), rcond=None)
        A0, B0 = coef
        yhat = 2**(A0 + B0*x)
        r = rmse_log2(y, yhat)
        # 局部等效倍率（近 n=0 的每轮比值）：f_eff = 2^{-k * ((1)^p - (0)^p)} = 2^{-k}，k=-B0
        k_local = -B0
        if (best is None) or (r < best["rmse_log2"]):
            best = dict(model="power_curvature", p=p, A=A0, B=B0, C=2**A0, k_local=k_local,
                        rmse_log2=r, yhat=yhat)
    return best

def run_search(y_obs):
    years = y_obs.index.to_list()
    n = np.arange(len(years), dtype=float)  # 0,1,2
    y = y_obs.values.astype(float)

    results = []
    # 1) 自由线性（base-2 指数衰减）
    results.append(fit_free_linear(n, y))
    # 2) 固定倍率对照
    for name, f in CANDIDATES.items():
        results.append(fit_fixed_factor(f, n, y) | {"label": name})
    # 3) 幂曲率扩展
    results.append(fit_power_curvature(n, y))

    # 整理与排序
    rows = []
    for r in results:
        row = {
            "model": r["model"],
            "rmse_log2": r["rmse_log2"],
            "C": r.get("C", np.nan),
            "k": r.get("k", np.nan),
            "p": r.get("p", np.nan),
            "k_local_at_n0": r.get("k_local", r.get("k", np.nan)),
            "factor_per_cycle_at_n0": float(2**(-r.get("k_local", r.get("k", np.nan)))) if ("k" in r or "k_local" in r) else np.nan,
        }
        if "label" in r: row["model"] = f"{row['model']}[{r['label']}]"
        rows.append(row)
    df = pd.DataFrame(rows).sort_values("rmse_log2").reset_index(drop=True)
    # 最优预测（按 rmse_log2 最小）
    best = results[np.argmin([r["rmse_log2"] for r in results])]
    yhat_best = best["yhat"]
    return df, n, y, yhat_best

def plot_metric(years, y, yhat, title, path_png):
    n = np.arange(len(years))
    plt.figure(figsize=(8.4,4.8))
    plt.plot(n, y, marker="o", linestyle="-", label="观测")
    plt.plot(n, yhat, marker="o", linestyle="--", label="最优拟合")
    plt.xticks(n, years)
    plt.xlabel("循环（按峰值年）"); plt.ylabel("AHR999 代表值")
    plt.title(title)
    plt.legend(); plt.tight_layout()
    plt.savefig(path_png, dpi=170); plt.close()

# ========= 主流程：对每个口径跑搜索 =========
writer = pd.ExcelWriter(OUTDIR / "ahr999_decay_search_all.xlsx", engine="openpyxl")
summary_rows = []

for metric_name, kw in METRICS:
    obs = collect_obs(metric_name, kw)
    models_df, n, y, yhat = run_search(obs["metric"])
    # 保存 sheet
    obs.to_excel(writer, sheet_name=f"{metric_name}_obs")
    models_df.to_excel(writer, sheet_name=f"{metric_name}_fits", index=False)

    # 汇总最优模型
    best_row = models_df.iloc[0].to_dict()
    best_row.update({
        "metric": metric_name,
        "y2017": float(y[0]), "y2021": float(y[1]), "y2025": float(y[2])
    })
    summary_rows.append(best_row)

    # 画图
    plot_metric(obs.index.tolist(), y, yhat,
                title=f"AHR999 跨轮衰减最优拟合（口径：{metric_name}）",
                path_png=OUTDIR / f"ahr999_decay_best_{metric_name}.png")

# 总结页
summary = pd.DataFrame(summary_rows).sort_values("rmse_log2")
summary.to_excel(writer, sheet_name="summary", index=False)
writer.close()

print("已输出：", (OUTDIR / "ahr999_decay_search_all.xlsx").resolve())
print("最优图像已保存到同目录（ahr999_decay_best_*.png）")
print("\nTop-1 汇总：")
print(summary.head(1))


C:\Users\nickc\AppData\Local\Temp\ipykernel_16388\3161480370.py:189: UserWarning: Glyph 24490 (\N{CJK UNIFIED IDEOGRAPH-5FAA}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_16388\3161480370.py:189: UserWarning: Glyph 29615 (\N{CJK UNIFIED IDEOGRAPH-73AF}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_16388\3161480370.py:189: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_16388\3161480370.py:189: UserWarning: Glyph 25353 (\N{CJK UNIFIED IDEOGRAPH-6309}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppData\Local\Temp\ipykernel_16388\3161480370.py:189: UserWarning: Glyph 23792 (\N{CJK UNIFIED IDEOGRAPH-5CF0}) missing from font(s) DejaVu Sans.
  plt.legend(); plt.tight_layout()
C:\Users\nickc\AppDa

已输出： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_decay_search_all.xlsx
最优图像已保存到同目录（ahr999_decay_best_*.png）

Top-1 汇总：
             model  rmse_log2          C   k     p  k_local_at_n0  \
0  power_curvature   0.001479  17.266925 NaN  0.72       2.374814   

   factor_per_cycle_at_n0           metric      y2017     y2021     y2025  
0                0.192801  peak_month_mean  17.257163  3.333878  1.146254  


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ========== 配置 ==========
IN_XLSX = Path("btc_fit_outputs/ahr999_daily.xlsx")
IN_CSV  = Path("btc_fit_outputs/ahr999_daily.csv")
OUTDIR  = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

# 三轮：减半 & 峰值（2017 高点按你修正：2017-12-18；2025 为假设）
HALVINGS = {2017: pd.Timestamp("2016-07-09"),
            2021: pd.Timestamp("2020-05-11"),
            2025: pd.Timestamp("2024-04-20")}
PEAKS    = {2017: pd.Timestamp("2017-12-18"),
            2021: pd.Timestamp("2021-11-10"),
            2025: pd.Timestamp("2025-08-15")}

GRID_POINTS = 200          # 归一化时间轴上的插值点数（越大越细）
USE_POWER_CURVE = False    # 若想试 n^p 曲率，把它改成 True
P_GRID = np.round(np.linspace(0.3, 2.0, 86), 2)  # 仅当上面为 True 时有效

# （可选）中文字体（Windows）
from matplotlib import rcParams, font_manager
for p in [r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\simhei.ttf"]:
    if Path(p).exists(): font_manager.fontManager.addfont(p)
rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False

# ========== 读取 AHR999 ==========
def load_ahr999(xlsx_path: Path, csv_path: Path) -> pd.DataFrame:
    if xlsx_path.exists():
        df = pd.read_excel(xlsx_path, sheet_name="AHR999")
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError("找不到 ahr999_daily.xlsx / .csv，请先生成。")
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    if "ahr999" not in df.columns:
        raise ValueError("缺少列 'ahr999'。")
    return df[["ahr999"]].rename(columns={"ahr999":"y"})

ahr = load_ahr999(IN_XLSX, IN_CSV)

def nearest(ts):
    ts = pd.Timestamp(ts)
    if ts in ahr.index: return ts
    before = ahr.index[ahr.index <= ts]
    after  = ahr.index[ahr.index >= ts]
    if len(before)>0: return before.max()
    return after.min()

# ========== 减半→峰值：取段 & 归一化时间 → 统一网格插值 ==========
def curve_halving_to_peak(year, m=GRID_POINTS):
    # 取区间
    h, p = nearest(HALVINGS[year]), nearest(PEAKS[year])
    seg = ahr.loc[h:p, "y"].copy()
    if seg.empty:
        raise ValueError(f"{year} 区间为空：{h}→{p}，请检查数据覆盖")

    # ---- 关键修改：用“天数 float”算归一化进度 u ∈ [0,1] ----
    # 分子：每个样本距 halving 的天数（浮点天）
    num_days = (seg.index - h) / pd.Timedelta(days=1)
    # 分母：peak 与 halving 的天数（浮点天）
    den_days = (p - h) / pd.Timedelta(days=1)
    u = (num_days / den_days).astype(float)

    # 去重 + 排序（防止重复日期或同一 u 值）
    df = pd.DataFrame({"u": u.values, "y": seg.values}).drop_duplicates("u").sort_values("u")

    # 统一网格插值到 m 个点
    grid = np.linspace(0.0, 1.0, m)
    y_on_grid = np.interp(grid, df["u"].values, df["y"].values)
    return grid, y_on_grid, h, p


years = [2017, 2021, 2025]
grid = None
Y = []   # 形如 [y2017(u), y2021(u), y2025(u)]
hp_rows = []
for y in years:
    g, yg, h, p = curve_halving_to_peak(y, GRID_POINTS)
    if grid is None: grid = g
    Y.append(yg)
    hp_rows.append({"year": y, "halving": h.date(), "peak": p.date(), "days_htp": int((p-h).days)})

Y = np.array(Y)  # shape = (3, GRID_POINTS)
hp_df = pd.DataFrame(hp_rows).set_index("year").sort_index()

# ========== 在 log2 域做“两因子”拟合：log2 Y = c - k * n (+ 可选 n^p) + S(u) ==========
def fit_two_factor_fullpath(Y, use_power=False, p_grid=P_GRID):
    """
    输入：Y[cycle, u] 为减半→峰值归一化后的 AHR999 曲线。
    返回：最优参数（k_hat、f_hat、c_hat、S_hat(u)）、以及拟合曲线 Y_hat、误差指标。
    """
    eps = 1e-9
    Z = np.log2(np.maximum(Y, eps))   # Z[cyc,u]
    cycles = Y.shape[0]
    n = np.arange(cycles, dtype=float)  # 0,1,2
    nbar = n.mean()

    def solve_for_k(p=1.0):
        # 1) 先用“每个 u 的均值”去掉形状：Zbar_u
        Zbar = Z.mean(axis=0)                        # (u,)
        Z_tilde = Z - Zbar[None, :]                  # (cyc,u)
        # 2) 对每个 cycle 求时间平均：m_y = mean_u Z_tilde
        m = Z_tilde.mean(axis=1)                     # (cyc,)
        # 3) 在线性模型 m_y ≈ a + b * (n^p) 上回归：b = slope = -k
        x = n**p
        X = np.vstack([np.ones_like(x), x]).T
        coef, *_ = np.linalg.lstsq(X, m, rcond=None)
        a, b = coef
        k_hat = -b
        c_prime = a                                  # 注意：这是“连同 mean(S) 的截距”
        # 4) 还原 S_hat(u)：Zbar_u = c' - k * nbar + S(u)  ⇒  S_hat(u) = Zbar_u - (c' - k*nbar)
        S_hat = Zbar - (c_prime - k_hat * nbar)
        # 5) 用 c' 与 S_hat 复原预测：Z_hat = c' - k*n + S_hat(u)
        Z_hat = (c_prime - k_hat * n[:,None]) + S_hat[None, :]
        Y_hat = 2.0 ** Z_hat
        # 6) 误差（log2 域 RMSE 更稳健）
        rmse_log2 = float(np.sqrt(np.mean((Z - Z_hat)**2)))
        rmse_abs  = float(np.sqrt(np.mean((Y - Y_hat)**2)))
        # 7) 给出“每轮倍率” f = 2^{-k}
        f_hat = 2.0 ** (-k_hat)
        return dict(p=p, k_hat=float(k_hat), f_hat=float(f_hat),
                    c_prime=float(c_prime), S_hat=S_hat, Y_hat=Y_hat,
                    rmse_log2=rmse_log2, rmse_abs=rmse_abs)

    if not use_power:
        return solve_for_k(p=1.0)
    else:
        # 搜索 p 以最小化 rmse_log2
        best = None
        for p in p_grid:
            res = solve_for_k(p=p)
            if best is None or res["rmse_log2"] < best["rmse_log2"]:
                best = res
        return best

res = fit_two_factor_fullpath(Y, use_power=USE_POWER_CURVE, p_grid=P_GRID)

k_hat = res["k_hat"]; f_hat = res["f_hat"]; p_used = res["p"]
Y_hat = res["Y_hat"]; S_hat = res["S_hat"]

# ========== 保存结果 ==========
# 1) 网格曲线（观测 & 拟合）
grid_df = pd.DataFrame({"u": grid})
for i, y in enumerate(years):
    grid_df[f"obs_{y}"] = Y[i]
    grid_df[f"fit_{y}"] = Y_hat[i]
grid_df.to_csv(OUTDIR / "ahr999_halving2peak_fullpath_grid.csv", index=False, encoding="utf-8-sig")

# 2) 参数与误差
params_df = pd.DataFrame({
    "param": ["k_hat","f_hat","p_used","rmse_log2","rmse_abs","n_bar"],
    "value": [k_hat, f_hat, p_used, res["rmse_log2"], res["rmse_abs"], np.mean(np.arange(len(years)))]
})
params_df.to_csv(OUTDIR / "ahr999_halving2peak_fit_params.csv", index=False, encoding="utf-8-sig")

# 3) 图：全路径拟合（原尺度）
plt.figure(figsize=(11.5,4.6))
for i, y in enumerate(years):
    plt.plot(grid, Y[i], label=f"{y} 观测")
    plt.plot(grid, Y_hat[i], linestyle="--", label=f"{y} 拟合")
plt.xlabel("归一化进度（减半→峰值）"); plt.ylabel("AHR999")
ttl = f"AHR999 减半→峰值 全路径拟合  |  最优 f=2^(-k)={f_hat:.3f} /轮"
if USE_POWER_CURVE: ttl += f"，p={p_used:.2f}"
plt.title(ttl)
plt.legend(ncol=3); plt.tight_layout()
plt.savefig(OUTDIR / "ahr999_halving2peak_fit.png", dpi=170); plt.close()

print("=== 拟合完成 ===")
print(f"k_hat = {k_hat:.4f}  →  每轮倍率 f_hat = 2^(-k) = {f_hat:.4f}")
if USE_POWER_CURVE: print(f"幂曲率 p = {p_used:.2f}")
print(f"RMSE_log2 = {res['rmse_log2']:.4f}，RMSE_abs = {res['rmse_abs']:.4f}")
print("输出：")
print("  - 网格曲线：", (OUTDIR / "ahr999_halving2peak_fullpath_grid.csv").resolve())
print("  - 参数文件：", (OUTDIR / "ahr999_halving2peak_fit_params.csv").resolve())
print("  - 拟合图：  ", (OUTDIR / "ahr999_halving2peak_fit.png").resolve())


=== 拟合完成 ===
k_hat = 0.2026  →  每轮倍率 f_hat = 2^(-k) = 0.8690
RMSE_log2 = 0.7830，RMSE_abs = 2.0264
输出：
  - 网格曲线： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_halving2peak_fullpath_grid.csv
  - 参数文件： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_halving2peak_fit_params.csv
  - 拟合图：   C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_halving2peak_fit.png


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ===== 配置：你只需要改这里几项 =====
IN_XLSX = Path("btc_fit_outputs/ahr999_daily.xlsx")
IN_CSV  = Path("btc_fit_outputs/ahr999_daily.csv")
OUTDIR  = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

HALVINGS = {2017: pd.Timestamp("2016-07-09"),
            2021: pd.Timestamp("2020-05-11"),
            2025: pd.Timestamp("2024-04-20")}
PEAKS    = {2017: pd.Timestamp("2017-12-18"),
            2021: pd.Timestamp("2021-11-10"),
            2025: pd.Timestamp("2025-08-15")}

GRID_POINTS = 200

# 要比较的设定（你可增删组合）
SETTINGS = [
    # (名字, u窗口(min,max), 仅用 AHR>thr?, 权重模式)
    ("uniform_full",   (0.0,1.0),  None,    "uniform"),  # 你的当前口径（对照）
    ("u_0.7_1.0",      (0.7,1.0),  None,    "uniform"),  # 只看接近峰值的后段
    ("ahr_gt_1_full",  (0.0,1.0),  1.0,     "uniform"),  # 只用 AHR>1 的点
    ("ahr_weight_full",(0.0,1.0),  None,    "ahr"),      # 用 AHR 自身作为权重
    ("ahr_weight_tail",(0.7,1.0),  None,    "ahr"),      # 尾段+热度加权
]

# ===== 读取 AHR999 =====
def load_ahr999(xlsx_path: Path, csv_path: Path) -> pd.DataFrame:
    if xlsx_path.exists():
        df = pd.read_excel(xlsx_path, sheet_name="AHR999")
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError("找不到 ahr999_daily.xlsx / .csv，请先生成。")
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    if "ahr999" not in df.columns:
        raise ValueError("缺少列 'ahr999'。")
    return df[["ahr999"]].rename(columns={"ahr999":"y"})

ahr = load_ahr999(IN_XLSX, IN_CSV)

def nearest(ts):
    ts = pd.Timestamp(ts)
    if ts in ahr.index: return ts
    before = ahr.index[ahr.index <= ts]
    after  = ahr.index[ahr.index >= ts]
    if len(before)>0: return before.max()
    return after.min()

def curve_halving_to_peak(year, m=GRID_POINTS):
    h, p = nearest(HALVINGS[year]), nearest(PEAKS[year])
    seg = ahr.loc[h:p, "y"].copy()
    if seg.empty: raise ValueError(f"{year} 区间为空：{h}→{p}")
    num_days = (seg.index - h) / pd.Timedelta(days=1)
    den_days = (p - h) / pd.Timedelta(days=1)
    u = (num_days / den_days).astype(float)
    df = pd.DataFrame({"u": u.values, "y": seg.values}).drop_duplicates("u").sort_values("u")
    grid = np.linspace(0.0, 1.0, m)
    y_on_grid = np.interp(grid, df["u"].values, df["y"].values)
    return grid, y_on_grid, h, p

years = [2017, 2021, 2025]
grid = None
Y = []
hp_rows = []
for y in years:
    g, yg, h, p = curve_halving_to_peak(y, GRID_POINTS)
    if grid is None: grid = g
    Y.append(yg)
    hp_rows.append({"year": y, "halving": h.date(), "peak": p.date(), "days_htp": int((p-h).days)})
Y = np.array(Y)  # (cycles, M)
hp_df = pd.DataFrame(hp_rows).set_index("year").sort_index()

# ===== 带权版“全路径”拟合：log2 Y = c - k*n + S(u)，但在 u 上做窗口/掩码/加权 =====
def fit_fullpath_with_weights(Y, grid, u_win=(0,1), thr=None, weight_mode="uniform"):
    eps = 1e-9
    Z = np.log2(np.maximum(Y, eps))   # (C,M)
    C, M = Z.shape
    n = np.arange(C, dtype=float)     # 0,1,2
    nbar = n.mean()

    # 构造窗口/掩码/权重
    umin, umax = u_win
    mask_u = (grid >= umin) & (grid <= umax)  # (M,)
    Zw = Z[:, mask_u]                          # (C,M_sel)
    Gw = grid[mask_u]
    Yw = (2**Zw)

    if thr is not None:
        mask_thr = (Yw > thr)                  # (C,M_sel)
    else:
        mask_thr = np.ones_like(Yw, dtype=bool)

    if weight_mode == "uniform":
        W = mask_thr.astype(float)
    elif weight_mode == "ahr":
        W = np.where(mask_thr, Yw, 0.0)        # AHR999 作为权重（>thr 才计权）
    else:
        raise ValueError("未知权重模式")

    # 防止全0
    if W.sum() == 0:
        raise ValueError("当前设定下无有效样本点（窗口太窄或阈值太高）")

    # 1) S(u)：先“跨cycle均值”去掉公共形状（按 cycle 无权，按 u 仅看有效点）
    #   注意：这里保持与无权版一致——S 由各周期等权平均得到
    Zbar = Zw.mean(axis=0)                     # (M_sel,)
    Ztil = Zw - Zbar[None, :]                  # (C,M_sel)

    # 2) 每个 cycle 的“加权平均残差” m_y（在 u 维度上按 W 权重平均）
    Wsum = W.sum(axis=1)                       # (C,)
    m = (Ztil * W).sum(axis=1) / Wsum          # (C,)

    # 3) 回归 m ≈ a + b*n → k = -b
    X = np.vstack([np.ones_like(n), n]).T
    coef, *_ = np.linalg.lstsq(X, m, rcond=None)
    a, b = coef
    k_hat = -float(b)
    c_prime = float(a)                         # 截距（包含常数偏移）

    # 4) 还原 S_hat(u)（与无权版一致的推导）
    S_hat = Zbar - (c_prime - k_hat * nbar)

    # 5) 预测并计算误差（既给“窗口内加权 RMSE_log2”，也给“全段无权 RMSE_log2”）
    Zhat_w = (c_prime - k_hat * n[:,None]) + S_hat[None, :]
    rmse_log2_weighted = float(np.sqrt((( (Zw - Zhat_w)**2 * (W / W.sum()) ).sum())))

    # 同时给一个“全段无权”的 RMSE（用于不同设定可比性）
    Z_full = Z
    S_full = np.interp(grid, Gw, S_hat)        # 插回全网格
    Zhat_full = (c_prime - k_hat * n[:,None]) + S_full[None, :]
    rmse_log2_full = float(np.sqrt(np.mean((Z_full - Zhat_full)**2)))

    return dict(k_hat=k_hat,
                f_hat=float(2**(-k_hat)),
                rmse_log2_weighted=rmse_log2_weighted,
                rmse_log2_full=rmse_log2_full,
                c_prime=c_prime,
                S_hat=S_hat, Gw=Gw,
                Zhat_full=Zhat_full)

# ===== 在多设定上搜索“最小误差”的 k =====
rows = []
best = None
for name, u_win, thr, wmode in SETTINGS:
    try:
        res = fit_fullpath_with_weights(Y, grid, u_win=u_win, thr=thr, weight_mode=wmode)
        row = dict(name=name, u_min=u_win[0], u_max=u_win[1], thr=thr, weight=wmode,
                   k_hat=round(res["k_hat"], 6), f_hat=round(res["f_hat"], 6),
                   rmse_log2_weighted=round(res["rmse_log2_weighted"], 6),
                   rmse_log2_full=round(res["rmse_log2_full"], 6))
        rows.append(row)
        if (best is None) or (res["rmse_log2_full"] < best["rmse_log2_full"]):
            best = (name, res)
    except Exception as e:
        rows.append(dict(name=name, u_min=u_win[0], u_max=u_win[1], thr=thr,
                         weight=wmode, k_hat=np.nan, f_hat=np.nan,
                         rmse_log2_weighted=np.nan, rmse_log2_full=np.nan, error=str(e)))

summary = pd.DataFrame(rows).sort_values("rmse_log2_full")
summary_path = OUTDIR / "ahr999_halving2peak_weighted_search_summary.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("搜索汇总：", summary_path.resolve())
print(summary)

# ===== 画“最优设定”的拟合对比图 =====
if best is not None:
    best_name, best_res = best
    Zhat_full = best_res["Zhat_full"]
    Yhat_full = 2**Zhat_full
    plt.figure(figsize=(11.5,4.6))
    for i, y in enumerate(years):
        plt.plot(grid, Y[i], label=f"{y} 观测")
        plt.plot(grid, Yhat_full[i], linestyle="--", label=f"{y} 拟合")
    ttl = f"最优设定：{best_name} | f=2^(-k)={best_res['f_hat']:.3f}  (k={best_res['k_hat']:.3f})"
    plt.title(ttl); plt.xlabel("归一化进度（减半→峰值）"); plt.ylabel("AHR999")
    plt.legend(ncol=3); plt.tight_layout()
    plt.savefig(OUTDIR / f"ahr999_halving2peak_fit_{best_name}.png", dpi=170); plt.close()
    print("最优图：", (OUTDIR / f"ahr999_halving2peak_fit_{best_name}.png").resolve())


搜索汇总： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_halving2peak_weighted_search_summary.csv
              name  u_min  u_max  thr   weight     k_hat     f_hat  \
0     uniform_full    0.0    1.0  NaN  uniform  0.202567  0.869003   
3    ahr_gt_1_full    0.0    1.0  1.0  uniform  0.578552  0.669635   
5  ahr_weight_full    0.0    1.0  NaN      ahr  0.696209  0.617192   
1        u_0.7_1.0    0.7    1.0  NaN  uniform  1.005441  0.498118   
7  ahr_weight_tail    0.7    1.0  NaN      ahr  1.227701  0.426997   
2        u_0.7_1.0    0.7    1.0  NaN  uniform       NaN       NaN   
4    ahr_gt_1_full    0.0    1.0  1.0  uniform       NaN       NaN   
6  ahr_weight_full    0.0    1.0  NaN      ahr       NaN       NaN   
8  ahr_weight_tail    0.7    1.0  NaN      ahr       NaN       NaN   

   rmse_log2_weighted  rmse_log2_full  \
0            0.782994        0.782994   
3            0.818649        0.841025   
5            1.054211        0.880644   
1            0.402817        1.177890   

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ========= 只改这里 =========
IN_XLSX = Path("btc_fit_outputs/ahr999_daily.xlsx")
IN_CSV  = Path("btc_fit_outputs/ahr999_daily.csv")
OUTDIR  = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

HALVINGS = {2017: pd.Timestamp("2016-07-09"),
            2021: pd.Timestamp("2020-05-11"),
            2025: pd.Timestamp("2024-04-20")}
PEAKS    = {2017: pd.Timestamp("2017-12-18"),   # 你修正的峰值
            2021: pd.Timestamp("2021-11-10"),
            2025: pd.Timestamp("2025-08-15")}   # 假设

GRID_POINTS = 200
U_WINDOW = (0.7, 1.0)      # 只看接近峰值的尾段，更关注“热度”
THRESH_AHR = None          # 设为 1.0 表示只用 AHR>1 的点；或 None 不限
WEIGHT_MODE = "ahr"        # "uniform" 或 "ahr"（用 AHR 自身做权重）
P_GRID = np.round(np.linspace(0.3, 2.0, 86), 2)  # 搜索 p（非线性弯曲度）

# ========= 读取 AHR999 =========
def load_ahr999(xlsx_path: Path, csv_path: Path) -> pd.DataFrame:
    if xlsx_path.exists():
        df = pd.read_excel(xlsx_path, sheet_name="AHR999")
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError("找不到 ahr999_daily.xlsx / .csv，请先生成。")
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    if "ahr999" not in df.columns:
        raise ValueError("缺少列 'ahr999'。")
    return df[["ahr999"]].rename(columns={"ahr999":"y"})

ahr = load_ahr999(IN_XLSX, IN_CSV)

def nearest(ts):
    ts = pd.Timestamp(ts)
    if ts in ahr.index: return ts
    before = ahr.index[ahr.index <= ts]
    after  = ahr.index[ahr.index >= ts]
    if len(before)>0: return before.max()
    return after.min()

def curve_halving_to_peak(year, m=GRID_POINTS):
    h, p = nearest(HALVINGS[year]), nearest(PEAKS[year])
    seg = ahr.loc[h:p, "y"].copy()
    if seg.empty: raise ValueError(f"{year} 区间为空：{h}→{p}")
    num_days = (seg.index - h) / pd.Timedelta(days=1)
    den_days = (p - h) / pd.Timedelta(days=1)
    u = (num_days / den_days).astype(float)
    df = pd.DataFrame({"u": u.values, "y": seg.values}).drop_duplicates("u").sort_values("u")
    grid = np.linspace(0.0, 1.0, m)
    y_on_grid = np.interp(grid, df["u"].values, df["y"].values)
    return grid, y_on_grid, h, p

years = [2017, 2021, 2025]
grid = None; Y = []; hp_rows = []
for y in years:
    g, yg, h, p = curve_halving_to_peak(y, GRID_POINTS)
    if grid is None: grid = g
    Y.append(yg); hp_rows.append({"year": y, "halving": h.date(), "peak": p.date(), "days_htp": int((p-h).days)})
Y = np.array(Y)  # (C, M)
hp_df = pd.DataFrame(hp_rows).set_index("year").sort_index()

# ========= 非线性跨轮拟合：log2 Y = c - k * n^p + S(u) =========
def fit_nonlinear_fullpath(Y, grid, u_win=(0,1), thr=None, weight_mode="uniform", p_grid=P_GRID):
    eps = 1e-9
    Z = np.log2(np.maximum(Y, eps))   # (C,M)
    C, M = Z.shape
    n = np.arange(C, dtype=float)     # 0,1,2
    nbar = n.mean()

    # 选尾段/阈值/权重
    umin, umax = u_win
    mask_u = (grid >= umin) & (grid <= umax)
    Zw = Z[:, mask_u]; Gw = grid[mask_u]
    Yw = (2**Zw)

    if thr is not None:
        mask_thr = (Yw > thr)
    else:
        mask_thr = np.ones_like(Yw, dtype=bool)

    if weight_mode == "uniform":
        W = mask_thr.astype(float)
    elif weight_mode == "ahr":
        W = np.where(mask_thr, Yw, 0.0)
    else:
        raise ValueError("未知权重模式")

    if W.sum() == 0:
        raise ValueError("无有效样本点（窗口太窄或阈值过高）")

    # 先去形状：按 cycle 等权平均得到公共形状 Zbar(u)
    Zbar = Zw.mean(axis=0)                 # (M_sel,)
    Ztil = Zw - Zbar[None, :]              # (C,M_sel)

    best = None
    for p in p_grid:
        x = n**p
        # 各 cycle 在 u 维度的加权平均残差 m_y
        Wsum = W.sum(axis=1)               # (C,)
        m = (Ztil * W).sum(axis=1) / Wsum  # (C,)
        # 回归 m ≈ a + b * x  → k = -b
        X = np.vstack([np.ones_like(x), x]).T
        coef, *_ = np.linalg.lstsq(X, m, rcond=None)
        a, b = coef
        k_hat = -float(b); c_prime = float(a)

        # 复原 S_hat(u) 并得到全网格预测
        S_hat = Zbar - (c_prime - k_hat * (nbar**p))
        Zhat_w = (c_prime - k_hat * (n[:,None]**p)) + S_hat[None, :]
        # 误差：给窗口加权 RMSE_log2 以及全段无权 RMSE_log2（可比）
        rmse_log2_weighted = float(np.sqrt((( (Zw - Zhat_w)**2 * (W / W.sum()) ).sum())))
        # 插回全网格
        S_full = np.interp(grid, Gw, S_hat)
        Zhat_full = (c_prime - k_hat * (n[:,None]**p)) + S_full[None, :]
        rmse_log2_full = float(np.sqrt(np.mean((np.log2(np.maximum(2**Z,eps)) - Zhat_full)**2)))

        cand = dict(p=p, k_hat=k_hat, c_prime=c_prime,
                    rmse_log2_weighted=rmse_log2_weighted, rmse_log2_full=rmse_log2_full,
                    Zhat_full=Zhat_full)
        if (best is None) or (cand["rmse_log2_full"] < best["rmse_log2_full"]):
            best = cand

    # 输出每轮有效倍率：f_0 = 2^{-k[(1)^p-(0)^p]}, f_1 = 2^{-k[(2)^p-(1)^p]}
    p = best["p"]; k = best["k_hat"]
    f0 = 2**(-k * ((1.0**p) - (0.0**p)))
    f1 = 2**(-k * ((2.0**p) - (1.0**p)))

    return dict(p_hat=p, k_hat=k, f0=f0, f1=f1,
                rmse_log2_full=best["rmse_log2_full"],
                rmse_log2_weighted=best["rmse_log2_weighted"],
                Zhat_full=best["Zhat_full"])

res = fit_nonlinear_fullpath(Y, grid, u_win=U_WINDOW, thr=THRESH_AHR, weight_mode=WEIGHT_MODE, p_grid=P_GRID)

# ========= 保存 & 画图 =========
params = {
    "u_min": U_WINDOW[0], "u_max": U_WINDOW[1],
    "thr": THRESH_AHR, "weight": WEIGHT_MODE,
    "p_hat": res["p_hat"], "k_hat": res["k_hat"],
    "f0 (2017→2021)": res["f0"], "f1 (2021→2025)": res["f1"],
    "rmse_log2_full": res["rmse_log2_full"],
    "rmse_log2_weighted": res["rmse_log2_weighted"],
}
pd.DataFrame(list(params.items()), columns=["param","value"]).to_csv(
    OUTDIR / "ahr999_halving2peak_nonlinear_params.csv", index=False, encoding="utf-8-sig"
)

# 网格数据（观测+拟合）
grid_df = pd.DataFrame({"u": grid})
for i, y in enumerate(years):
    grid_df[f"obs_{y}"] = Y[i]
    grid_df[f"fit_{y}"] = (2**res["Zhat_full"][i])
grid_df.to_csv(OUTDIR / "ahr999_halving2peak_nonlinear_grid.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(11.5,4.6))
for i, y in enumerate(years):
    plt.plot(grid, Y[i], label=f"{y} 观测")
    plt.plot(grid, 2**res["Zhat_full"][i], "--", label=f"{y} 拟合")
ttl = (f"非线性跨轮：log2 Y ≈ c - k·n^p + S(u) | "
       f"p={res['p_hat']:.2f}, k={res['k_hat']:.3f}, "
       f"f0={res['f0']:.3f}, f1={res['f1']:.3f}  [{WEIGHT_MODE}, u∈{U_WINDOW}]")
plt.title(ttl); plt.xlabel("归一化进度（减半→峰值）"); plt.ylabel("AHR999")
plt.legend(ncol=3); plt.tight_layout()
plt.savefig(OUTDIR / "ahr999_halving2peak_nonlinear_fit.png", dpi=170); plt.close()

print("已保存：")
print("  参数：", (OUTDIR / "ahr999_halving2peak_nonlinear_params.csv").resolve())
print("  网格：", (OUTDIR / "ahr999_halving2peak_nonlinear_grid.csv").resolve())
print("  图像：", (OUTDIR / "ahr999_halving2peak_nonlinear_fit.png").resolve())
print("\n最关键：p_hat（非线性弯曲度）、k_hat；以及每一轮的有效倍率 f0 / f1（不相等 → 非线性）。")


已保存：
  参数： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_halving2peak_nonlinear_params.csv
  网格： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_halving2peak_nonlinear_grid.csv
  图像： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_halving2peak_nonlinear_fit.png

最关键：p_hat（非线性弯曲度）、k_hat；以及每一轮的有效倍率 f0 / f1（不相等 → 非线性）。


In [11]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# ========= 可调参数 =========
IN_XLSX = Path("btc_fit_outputs/ahr999_daily.xlsx")
IN_CSV  = Path("btc_fit_outputs/ahr999_daily.csv")
OUTDIR  = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

PEAKS = {2017: pd.Timestamp("2017-12-18"),
         2021: pd.Timestamp("2021-11-10")}
M_REF        = 500      # 统一参考长度：峰前 500 天
ROLL_WIN     = 28       # 动态波动退火：滚动均值/方差窗口（天）
DTW_BAND     = 0.08     # Sakoe–Chiba 带宽（占模板长度比例）
LAMBDA_GRID  = np.linspace(0, 1, 21)   # 模板混合权 λ
GUARD_TPL_END= 40       # 不允许对齐到模板最后 N 天（防“贴边”）
TAIL_SLOPE_WIN= 20      # 末端斜率估计窗口（点数）
PEAK_WEIGHT_ALPHA = 2.0 # 峭度加权强度（越大越强调“波峰”重合）
EPS = 1e-9

# ========= 数据读取 =========
def load_ahr999(xlsx, csv):
    if xlsx.exists():
        df = pd.read_excel(xlsx, sheet_name="AHR999")
    elif csv.exists():
        df = pd.read_csv(csv)
    else:
        raise FileNotFoundError("找不到 ahr999_daily.xlsx/.csv")
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    return df[["ahr999"]].rename(columns={"ahr999":"y"})

ahr = load_ahr999(IN_XLSX, IN_CSV)

# ========= 构造“峰前 500 天/最近 500 天” =========
def seg_pre_peak(year, m=M_REF):
    pk = PEAKS[year]
    lo = pk - pd.Timedelta(days=m-1)
    s = ahr.loc[lo:pk, "y"].asfreq("D").ffill()
    assert len(s)>=m-10, f"{year} 覆盖不足"
    return s.index, s.values

def seg_recent_25(m=M_REF):
    hi = ahr.index.max()
    lo = hi - pd.Timedelta(days=m-1)
    s = ahr.loc[lo:hi, "y"].asfreq("D").ffill()
    return s.index, s.values

t17, v17 = seg_pre_peak(2017, M_REF)
t21, v21 = seg_pre_peak(2021, M_REF)
t25, v25 = seg_recent_25(M_REF)

# ========= 动态波动退火（滚动 z-score） =========
def roll_z(x, win=ROLL_WIN):
    z = np.log2(np.maximum(x, EPS))
    s = pd.Series(z)
    mu = s.rolling(win, center=True, min_periods=win//2).mean().bfill().ffill()
    sd = s.rolling(win, center=True, min_periods=win//2).std(ddof=0).bfill().ffill()
    sd = sd.replace(0, sd.mean() or 1.0)
    zscore = (z - mu.values) / sd.values
    return zscore, z  # (z-score, 原始log2)

z17, Z17 = roll_z(v17)
z21, Z21 = roll_z(v21)
z25, Z25 = roll_z(v25)

# ========= 无权 DTW，把 2021 对齐到 2017（获得统一模板网格） =========
def dtw_path(x, y, band):
    n, m = len(x), len(y)
    band = max(1, int(band*m))
    INF = 1e18
    D = np.full((n+1, m+1), INF); D[0,0]=0.0
    for i in range(1,n+1):
        j0, j1 = max(1, i-band), min(m, i+band)
        xi = x[i-1]
        for j in range(j0, j1+1):
            d = (xi - y[j-1])**2
            D[i,j] = d + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
    i,j=n,m; pi,pj=[],[]
    while i>0 and j>0:
        pi.append(i-1); pj.append(j-1)
        i,j=min(((i-1,j),(i,j-1),(i-1,j-1)), key=lambda t:D[t])
    return np.array(pi[::-1]), np.array(pj[::-1]), D[n,m]

# 2021→2017 的映射
pi_21, pj_17, _ = dtw_path(z21, z17, DTW_BAND)

def warp_to_ref(src, pi_src, pj_ref, m_ref):
    # 把 src[•] 通过路径 (pi_src -> pj_ref) 聚合到 0..m_ref-1 网格
    vals = [[] for _ in range(m_ref)]
    for i,j in zip(pi_src, pj_ref):
        vals[j].append(src[i])
    out = np.array([np.mean(v) if v else np.nan for v in vals])
    # 缺失用邻近插值
    s = pd.Series(out).interpolate(limit_direction="both").bfill().ffill()
    return s.values

# 在 2017 网格上得到：2021 的 z-score & log2，以及二者的混合模板
z21_on17 = warp_to_ref(z21, pi_21, pj_17, M_REF)
Z21_on17 = warp_to_ref(Z21, pi_21, pj_17, M_REF)

# ========= 计算梯度权重（强调局部峰） =========
def peak_weights(curve_z):
    g = np.gradient(curve_z)               # 一阶差分
    p = np.abs(g)
    p /= (np.percentile(p, 95) + 1e-9)     # 归一
    w = 1.0 + PEAK_WEIGHT_ALPHA * np.clip(p, 0, 1)
    return w

# ========= 搜索 λ：把 2025 对齐到混合模板（带权 DTW） =========
def dtw_weighted(x, y, w, band, m_guard=0):
    n, m = len(x), len(y)
    m_eff = m - m_guard
    band = max(1, int(band*m_eff))
    INF = 1e18
    D = np.full((n+1, m_eff+1), INF); D[0,0]=0.0
    for i in range(1,n+1):
        j0, j1 = max(1, i-band), min(m_eff, i+band)
        xi = x[i-1]
        for j in range(j0, j1+1):
            d = w[j-1] * (xi - y[j-1])**2     # 关键：按模板位置加权
            D[i,j] = d + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
    i,j=n,m_eff; pi,pj=[],[]
    while i>0 and j>0:
        pi.append(i-1); pj.append(j-1)
        i,j=min(((i-1,j),(i,j-1),(i-1,j-1)), key=lambda t:D[t])
    return np.array(pi[::-1]), np.array(pj[::-1]), D[n,m_eff], m_eff

best=None
for lam in LAMBDA_GRID:
    S_mix = lam*z17 + (1-lam)*z21_on17       # z-score 模板
    L_mix = lam*Z17 + (1-lam)*Z21_on17       # 原 log2 模板
    w = peak_weights(S_mix)
    pi, pj, dist, m_eff = dtw_weighted(z25, S_mix, w, DTW_BAND, m_guard=GUARD_TPL_END)
    if (best is None) or (dist < best["dist"]):
        best = dict(lam=lam, pi=pi, pj=pj, dist=dist, S_mix=S_mix, L_mix=L_mix, m_eff=m_eff)

lam = best["lam"]; pi = best["pi"]; pj = best["pj"]
S_mix = best["S_mix"]; L_mix = best["L_mix"]; M_eff = best["m_eff"]

# ========= 剩余天数与峰值 AHR =========
# 末端斜率（模板步/观测步）
win = min(TAIL_SLOPE_WIN, len(pi))
di = (pi[-1] - pi[-win]) + 1e-9
dj = (pj[-1] - pj[-win]) + 1e-9
slope = dj / di

j_end = pj[-1]
remain_tpl = (M_REF - 1) - j_end
days_to_peak = remain_tpl / max(slope, 1e-6)
pred_peak_date = t25[-1] + pd.Timedelta(days=days_to_peak)

# 幅度回归：Z25 ≈ a + s*L_mix
Zmix_warp = L_mix[pj]
Z25_warp  = Z25[pi]
X = np.vstack([np.ones_like(Zmix_warp), Zmix_warp]).T
a_hat, s_hat = np.linalg.lstsq(X, Z25_warp, rcond=None)[0]
Z_peak_est = a_hat + s_hat * L_mix[-1]
ahr_peak_est = float(2**Z_peak_est)

# ========= 导出 & 画图 =========
summary = pd.DataFrame({
    "param": ["lambda","dtw_dist","tail_slope","j_end","remain_tpl",
              "days_to_peak","pred_peak_date","a_hat","s_hat","ahr_peak_est"],
    "value": [lam, best["dist"], slope, int(j_end), int(remain_tpl),
              float(days_to_peak), pred_peak_date.date(), a_hat, s_hat, ahr_peak_est]
})
summary.to_csv(OUTDIR / "ahr999_overlap_dynamic_vol_nowcast.csv", index=False, encoding="utf-8-sig")

u_obs = pj / (M_REF - 1)
vmix = 2**L_mix

plt.figure(figsize=(12.2,4.8))
plt.plot(u_obs, v25[pi], label="2025（映射到模板u）")
plt.plot(np.linspace(0,1,M_REF), vmix, "--", label=f"2017/2021 混合模板 λ={lam:.2f}")
plt.axvline(1.0, ls="--", c="gray", lw=1)
plt.scatter([1.0],[ahr_peak_est], marker="*", s=130, label=f"预测峰值 AHR={ahr_peak_est:.3f}")
plt.xlabel("模板进度 u（0→峰值）"); plt.ylabel("AHR999")
ttl = f"三轮重叠（动态波动退火 + 加权DTW）：剩余≈{days_to_peak:.0f}天 → {pred_peak_date.date()}"
plt.title(ttl); plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "ahr999_overlap_dynamic_vol_nowcast.png", dpi=170); plt.close()

print("OK：", (OUTDIR / "ahr999_overlap_dynamic_vol_nowcast.csv").resolve())
print("图：", (OUTDIR / "ahr999_overlap_dynamic_vol_nowcast.png").resolve())


OK： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_overlap_dynamic_vol_nowcast.csv
图： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_overlap_dynamic_vol_nowcast.png


In [12]:
# === 三轮同时对齐（不混合）、动态波动退火、前置波峰优先 ===
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# ========= 配置（只改这里）=========
IN_XLSX = Path("btc_fit_outputs/ahr999_daily.xlsx")
IN_CSV  = Path("btc_fit_outputs/ahr999_daily.csv")
OUTDIR  = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

PEAKS = {2017: pd.Timestamp("2017-12-18"),
         2021: pd.Timestamp("2021-11-10")}        # 历史真的peak
M_REF        = 500     # 统一参考长度：看“peak前500天”
ROLL_WIN     = 28      # 动态波动退火：滚动z-score窗口（天）
DTW_BAND     = 0.08    # DTW带宽（占模板长度比例）
GUARD_END    = 40      # 不允许贴到模板末端最后 N 天
PEAK_W_ALPHA = 2.5     # 峰权重强度（强调对齐波峰）
TAIL_WIN     = 20      # 末端斜率窗口（估“剩余天数”）
LOCAL_WIN_U  = 0.05    # 局部回归的u窗口半宽（估 a(u), b(u)）
MARGIN_B     = 0.02    # 单调约束余量：b2017≥b2021+margin≥b2025+margin

# ========= 读取 AHR999 =========
def load_ahr999(xlsx, csv):
    if xlsx.exists():
        df = pd.read_excel(xlsx, sheet_name="AHR999")
    elif csv.exists():
        df = pd.read_csv(csv)
    else:
        raise FileNotFoundError("找不到 ahr999_daily.xlsx/.csv")
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    if "ahr999" not in df.columns:
        raise ValueError("缺少列 'ahr999'")
    return df[["ahr999"]].rename(columns={"ahr999":"y"})

ahr = load_ahr999(IN_XLSX, IN_CSV)

# ========= 构造三个片段 =========
def seg_pre_peak(year, m=M_REF):
    pk = PEAKS[year]
    lo = pk - pd.Timedelta(days=m-1)
    s = ahr.loc[lo:pk, "y"].asfreq("D").ffill()
    return s.index, s.values

def seg_recent_25(m=M_REF):
    hi = ahr.index.max()
    lo = hi - pd.Timedelta(days=m-1)
    s = ahr.loc[lo:hi, "y"].asfreq("D").ffill()
    return s.index, s.values

t17, v17 = seg_pre_peak(2017, M_REF)
t21, v21 = seg_pre_peak(2021, M_REF)
t25, v25 = seg_recent_25(M_REF)   # 2025 不知道peak，只取最近500天

# ========= 动态波动退火：滚动z-score（每日/每月都平滑变化）=========
EPS = 1e-9
def roll_z(x, win=ROLL_WIN):
    z = np.log2(np.maximum(x, EPS))
    s = pd.Series(z)
    mu = s.rolling(win, center=True, min_periods=win//2).mean().bfill().ffill()
    sd = s.rolling(win, center=True, min_periods=win//2).std(ddof=0).bfill().ffill()
    sd = sd.replace(0, sd.mean() or 1.0)
    zscore = (z - mu.values) / sd.values
    return zscore, z   # (z-score, 原始log2)

z17, Z17 = roll_z(v17)
z21, Z21 = roll_z(v21)
z25, Z25 = roll_z(v25)

# ========= 加权DTW（带带宽与末端保护）=========
def dtw_weighted(x, y, w, band, guard_end=0):
    n, m = len(x), len(y)
    m_eff = max(10, m - guard_end)  # 不让贴到末端
    band = max(1, int(band*m_eff))
    INF = 1e18
    D = np.full((n+1, m_eff+1), INF); D[0,0]=0.0
    for i in range(1, n+1):
        j0, j1 = max(1, i-band), min(m_eff, i+band)
        xi = x[i-1]
        for j in range(j0, j1+1):
            d = w[j-1] * (xi - y[j-1])**2
            D[i,j] = d + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
    # 回溯
    i,j = n, m_eff
    pi, pj = [], []
    while i>0 and j>0:
        pi.append(i-1); pj.append(j-1)
        i,j = min(((i-1,j),(i,j-1),(i-1,j-1)), key=lambda t: D[t])
    return np.array(pi[::-1]), np.array(pj[::-1]), D[n,m_eff], m_eff

# ========= 峰权重（强调对齐前置波峰）=========
def peak_weights(curve_z):
    g1 = np.gradient(curve_z)
    curv = np.abs(g1)
    curv /= (np.percentile(curv, 95) + 1e-9)
    return 1.0 + PEAK_W_ALPHA * np.clip(curv, 0, 1)

# ========= 以2017为“时间参考”，对齐 2021 和 2025（不混合）=========
w17 = peak_weights(z17)
pi21, pj21, dist21, m_eff = dtw_weighted(z21, z17, w17, DTW_BAND, guard_end=GUARD_END)
pi25, pj25, dist25, _     = dtw_weighted(z25, z17, w17, DTW_BAND, guard_end=GUARD_END)

u_ref = np.linspace(0, 1, M_REF)             # 参考u（2017）
u21 = pj21 / (M_REF - 1)                     # 2021→2017 的映射u
u25 = pj25 / (M_REF - 1)                     # 2025→2017 的映射u

# ========= 局部回归：随u缓慢变化的 a_c(u), b_c(u) =========
# 在对齐点附近做 OLS：Z_c ≈ a(u) + b(u) * Z_2017
def local_ab(u_obs, Z_ref_at_pj, Z_obs_at_pi, u_grid, win=LOCAL_WIN_U):
    a = np.full_like(u_grid, np.nan, dtype=float)
    b = np.full_like(u_grid, np.nan, dtype=float)
    for k, ug in enumerate(u_grid):
        mask = (np.abs(u_obs - ug) <= win)
        if mask.sum() >= 12:  # 至少12个点
            X = np.vstack([np.ones(mask.sum()), Z_ref_at_pj[mask]]).T
            y = Z_obs_at_pi[mask]
            aa, bb = np.linalg.lstsq(X, y, rcond=None)[0]
            a[k], b[k] = aa, bb
    # 平滑
    a = pd.Series(a).interpolate().bfill().ffill().rolling(11, center=True, min_periods=3).mean().bfill().ffill().values
    b = pd.Series(b).interpolate().bfill().ffill().rolling(11, center=True, min_periods=3).mean().bfill().ffill().values
    return a, b

# 取对齐路径上的 Z_2017、Z_2021、Z_2025
Z17_on21 = Z17[pj21]; Z21_on21 = Z21[pi21]
Z17_on25 = Z17[pj25]; Z25_on25 = Z25[pi25]

a21, b21 = local_ab(u21, Z17_on21, Z21_on21, u_ref, win=LOCAL_WIN_U)
a25, b25 = local_ab(u25, Z17_on25, Z25_on25, u_ref, win=LOCAL_WIN_U)

# ========= 单调约束：2017 ≥ 2021 ≥ 2025（幅度缓慢递减）=========
b21 = np.minimum(b21, 1.0 - MARGIN_B)
b25 = np.minimum(b25, b21 - MARGIN_B)

# ========= 估“剩余天数”：用 2025 路径末端斜率 =========
win = min(TAIL_WIN, len(pi25))
di = (pi25[-1] - pi25[-win]) + 1e-9
dj = (pj25[-1] - pj25[-win]) + 1e-9
slope_25 = dj / di
j_end = pj25[-1]
remain_tpl = (M_REF - 1) - j_end
days_to_peak = remain_tpl / max(slope_25, 1e-6)
pred_peak_date = t25[-1] + pd.Timedelta(days=days_to_peak)

# ========= 估“峰值AHR”：在 u=1 用 a25(1), b25(1), Z17(1) =========
a25_1 = a25[-1]; b25_1 = b25[-1]; Z17_1 = Z17[-1]
Z_peak_25 = a25_1 + b25_1 * Z17_1
ahr_peak_25 = float(2**Z_peak_25)

# ========= 可视化：三轮都映射到同一u轴，比较“前置波峰” =========
# 2017（参考，原幅度）
y17_ref = 2**Z17
# 2021、2025（用各自 a(u), b(u) 投影到2017的幅度框架，观察重叠程度）
y21_proj = 2**(a21 + b21 * Z17)
y25_proj = 2**(a25 + b25 * Z17)

plt.figure(figsize=(12.8,4.8))
plt.plot(u_ref, y17_ref, label="2017（参考）")
plt.plot(u_ref, y21_proj, label="2021（动态退火对齐）")
plt.plot(u_ref, y25_proj, label="2025（动态退火对齐）")
plt.axvline(1.0, ls="--", c="gray", lw=1)
plt.scatter([1.0],[ahr_peak_25], marker="*", s=130, label=f"2025 预测峰 AHR={ahr_peak_25:.3f}")
plt.xlabel("统一进度 u（0→peak）"); plt.ylabel("AHR999")
plt.title(f"三轮同轴对齐（不混合）｜动态波动退火 + 局部回归｜剩余≈{days_to_peak:.0f}天 → {pred_peak_date.date()}")
plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "ahr999_three_overlaps_dynamic.png", dpi=170); plt.close()

# ========= 导出关键结果 =========
res = pd.DataFrame({
    "param": ["dtw_dist_2021","dtw_dist_2025","slope_tail_2025","j_end_2025","remain_tpl",
              "days_to_peak","pred_peak_date","ahr_peak_2025"],
    "value": [dist21, dist25, slope_25, int(j_end), int(remain_tpl),
              float(days_to_peak), pred_peak_date.date(), ahr_peak_25]
})
res.to_csv(OUTDIR / "ahr999_three_overlaps_dynamic_metrics.csv", index=False, encoding="utf-8-sig")

# 也把 a(u), b(u) 导出来，便于你看随时间的缓慢衰减
df_ab = pd.DataFrame({
    "u": u_ref,
    "a_2021": a21, "b_2021": b21,
    "a_2025": a25, "b_2025": b25
})
df_ab.to_csv(OUTDIR / "ahr999_local_ab_profiles.csv", index=False, encoding="utf-8-sig")

print("输出：")
print("  图：", (OUTDIR / "ahr999_three_overlaps_dynamic.png").resolve())
print("  指标：", (OUTDIR / "ahr999_three_overlaps_dynamic_metrics.csv").resolve())
print("  a(u), b(u) 曲线：", (OUTDIR / "ahr999_local_ab_profiles.csv").resolve())


输出：
  图： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_three_overlaps_dynamic.png
  指标： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_three_overlaps_dynamic_metrics.csv
  a(u), b(u) 曲线： C:\Users\nickc\yanda\btc\btc_fit_outputs\ahr999_local_ab_profiles.csv


In [14]:
# === 三轮同时对齐（不混合）+ 迭代“动态波动退火”，优先重叠前置波峰 ===
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# ========= 可调参数 =========
IN_XLSX = Path("btc_fit_outputs/ahr999_daily.xlsx")
IN_CSV  = Path("btc_fit_outputs/ahr999_daily.csv")
OUTDIR  = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

PEAKS = {2017: pd.Timestamp("2017-12-18"),
         2021: pd.Timestamp("2021-11-10")}
M_REF        = 500      # 统一参考长度：看“峰前500天”
ROLL_WIN     = 28       # 滚动z-score窗口（动态退火的基准）
DTW_BAND     = 0.06     # DTW带宽（占模板长度比例，越小越“保形”）
GUARD_END    = 60       # 不允许对齐到模板最后 N 天（防贴边）
PEAK_W_ALPHA = 3.2      # 峰权重强度（强调波峰重叠）
TAIL_WIN     = 20       # 末端斜率窗口（估“剩余天数”）
LOCAL_WIN_U  = 0.06     # 局部回归的u窗口半宽（估 a(u), b(u)）
VOL_TARGET_MULT = 1.25  # << 目标波动曲线的整体放大倍数（>1 会“放大 2025 的波动”，便于观察/拟合）
SCALE_CLIP   = (0.4, 3.0) # 按日缩放的上下限，防炸

# ========= 基础工具 =========
EPS = 1e-9
def load_ahr999(xlsx, csv):
    if xlsx.exists():  df = pd.read_excel(xlsx, sheet_name="AHR999")
    elif csv.exists(): df = pd.read_csv(csv)
    else: raise FileNotFoundError("找不到 ahr999_daily.xlsx/.csv")
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    if "ahr999" not in df.columns: raise ValueError("缺少列 'ahr999'")
    return df[["ahr999"]].rename(columns={"ahr999":"y"})

ahr = load_ahr999(IN_XLSX, IN_CSV)

def seg_pre_peak(year, m=M_REF):
    pk = PEAKS[year]; lo = pk - pd.Timedelta(days=m-1)
    s = ahr.loc[lo:pk, "y"].asfreq("D").ffill()
    return s.index, s.values

def seg_recent_25(m=M_REF):
    hi = ahr.index.max(); lo = hi - pd.Timedelta(days=m-1)
    s = ahr.loc[lo:hi, "y"].asfreq("D").ffill()
    return s.index, s.values

t17, v17 = seg_pre_peak(2017, M_REF)
t21, v21 = seg_pre_peak(2021, M_REF)
t25, v25 = seg_recent_25(M_REF)

def roll_stats(x, win=ROLL_WIN):
    z = np.log2(np.maximum(x, EPS))
    s = pd.Series(z)
    mu = s.rolling(win, center=True, min_periods=win//2).mean().bfill().ffill().values
    sd = s.rolling(win, center=True, min_periods=win//2).std(ddof=0).bfill().ffill().values
    sd = np.where(sd==0, np.nan, sd)
    sd = pd.Series(sd).fillna(np.nanmedian(sd) if np.isfinite(np.nanmedian(sd)) else 1.0).values
    return z, mu, sd

Z17, MU17, SD17 = roll_stats(v17)
Z21, MU21, SD21 = roll_stats(v21)
Z25, MU25, SD25 = roll_stats(v25)

# ========= 目标“波动曲线” σ*(u)：由 2017/2021 的滚动std在 u 轴上的均值 + 单调递减约束 =========
u_ref = np.linspace(0,1,M_REF)
def to_u_series(sd):
    # sd 已按峰前500天切好，直接线性映射到 u
    return pd.Series(sd, index=u_ref)

sigma_target = (to_u_series(SD17) + to_u_series(SD21)) / 2.0
# 平滑 & 单调递减：先移动平均，再从右向左取累积最小
sigma_target = sigma_target.rolling(21, center=True, min_periods=5).mean()
sigma_target = sigma_target.bfill().ffill()
sigma_target = sigma_target[::-1].cummin()[::-1]
sigma_target = (sigma_target * VOL_TARGET_MULT).values  # 整体放大（可>1）

# ========= 加权DTW（不混合） =========
def peak_weights_from_ref(z_ref):
    g = np.gradient(z_ref)
    p = np.abs(g); p /= (np.percentile(p, 95) + 1e-9)
    return 1.0 + PEAK_W_ALPHA * np.clip(p, 0, 1)

def dtw_weighted(x, y, w, band, guard_end=0):
    n, m = len(x), len(y)
    m_eff = max(10, m-guard_end); band = max(1, int(band*m_eff)); INF=1e18
    D = np.full((n+1, m_eff+1), INF); D[0,0]=0.0
    for i in range(1,n+1):
        j0, j1 = max(1, i-band), min(m_eff, i+band)
        xi = x[i-1]
        for j in range(j0, j1+1):
            d = w[j-1]*(xi - y[j-1])**2
            D[i,j] = d + min(D[i-1,j], D[i,j-1], D[i-1,j-1])
    i,j=n,m_eff; pi,pj=[],[]
    while i>0 and j>0:
        pi.append(i-1); pj.append(j-1)
        i,j = min(((i-1,j),(i,j-1),(i-1,j-1)), key=lambda t: D[t])
    return np.array(pi[::-1]), np.array(pj[::-1]), D[n,m_eff]

# ========= 一次“按日幅度缩放”的函数（给定 u 映射） =========
def amplitude_rescale(Z, MU, SD, u_map):
    # 围绕滚动均值按 σ*(u)/SD(t) 放大/压缩
    sig_u = np.interp(u_map, u_ref, sigma_target)
    scale = np.clip(sig_u / SD, SCALE_CLIP[0], SCALE_CLIP[1])
    Z_new = MU + (Z - MU) * scale
    return Z_new, scale

# ========= 迭代两轮：退火→对齐→更新u→再退火→再对齐 =========
def align_once(Z17_cur, Z21_cur, Z25_cur):
    # 以 2017 为参考；对齐对象用 z-score（去均值除std后的形状）
    def zscore_from(Z):
        s = pd.Series(Z)
        mu = s.rolling(ROLL_WIN, center=True, min_periods=ROLL_WIN//2).mean().bfill().ffill()
        sd = s.rolling(ROLL_WIN, center=True, min_periods=ROLL_WIN//2).std(ddof=0).bfill().ffill()
        sd = sd.replace(0, sd.median() or 1.0)
        return ((s-mu)/sd).values, mu.values, sd.values
    z17,_,_ = zscore_from(Z17_cur)
    z21,_,_ = zscore_from(Z21_cur)
    z25,_,_ = zscore_from(Z25_cur)
    w17 = peak_weights_from_ref(z17)
    pi21, pj21, _ = dtw_weighted(z21, z17, w17, DTW_BAND, guard_end=GUARD_END)
    pi25, pj25, _ = dtw_weighted(z25, z17, w17, DTW_BAND, guard_end=GUARD_END)
    u21 = pj21/(M_REF-1); u25 = pj25/(M_REF-1)
    return (pi21,pj21,u21),(pi25,pj25,u25)
def u_map_per_index(pi, pj, n_src, m_ref):
    """
    把 DTW 路径 (pi, pj) 转成“每个原始 i 的 u 映射”，长度 = n_src。
    u[i] = mean( pj | pi==i ) / (m_ref-1) ，缺失用插值/前后填充。
    """
    buckets = [[] for _ in range(n_src)]
    for ii, jj in zip(pi, pj):
        buckets[ii].append(jj)
    j_mean = np.array([np.mean(b) if b else np.nan for b in buckets], dtype=float)
    j_mean = pd.Series(j_mean).interpolate().bfill().ffill().values
    return j_mean / (m_ref - 1)


# 初始 u（线性）
u21_guess = u_ref.copy()
u25_guess = u_ref.copy()

(PI21_1, PJ21_1, _u21_path), (PI25_1, PJ25_1, _u25_path) = align_once(Z17_1, Z21_1, Z25_1)

# 把路径上的配对 (pi, pj) 转成“每个原始 i 的 u 映射”（长度=500）
u21_map1 = u_map_per_index(PI21_1, PJ21_1, n_src=len(Z21), m_ref=M_REF)
u25_map1 = u_map_per_index(PI25_1, PJ25_1, n_src=len(Z25), m_ref=M_REF)

# 第2轮退火（用 per-index 的 u 映射做按日幅度缩放）
Z21_2, scale21 = amplitude_rescale(Z21, MU21, SD21, u21_map1)
Z25_2, scale25 = amplitude_rescale(Z25, MU25, SD25, u25_map1)


(PI21_1,PJ21_1,u21_1),(PI25_1,PJ25_1,u25_1) = align_once(Z17_1, Z21_1, Z25_1)

# 第2轮退火（用对齐得到的 u 再做一次放大/压缩）
Z21_2, scale21 = amplitude_rescale(Z21, MU21, SD21, u21_1)
Z25_2, scale25 = amplitude_rescale(Z25, MU25, SD25, u25_1)

(PI21,PJ21,u21),(PI25,PJ25,u25) = align_once(Z17, Z21_2, Z25_2)

# ========= 末端斜率 → 剩余天数 =========
def tail_slope(pi, pj, win=TAIL_WIN):
    win = min(win, len(pi)); di = (pi[-1]-pi[-win])+1e-9; dj=(pj[-1]-pj[-win])+1e-9
    return dj/di, pj[-1]

slope25, j_end25 = tail_slope(PI25, PJ25, TAIL_WIN)
remain_tpl = (M_REF-1) - j_end25
days_to_peak = remain_tpl / max(slope25, 1e-6)
pred_peak_date = t25[-1] + pd.Timedelta(days=days_to_peak)

# ========= 局部回归：在统一u轴上估 a(u), b(u)（不混合） =========
def local_ab(u_obs, Z_ref_at_pj, Z_obs_at_pi, u_grid, win=LOCAL_WIN_U):
    a = np.full_like(u_grid, np.nan, dtype=float)
    b = np.full_like(u_grid, np.nan, dtype=float)
    for k, ug in enumerate(u_grid):
        mask = (np.abs(u_obs - ug) <= win)
        if mask.sum() >= 12:
            X = np.vstack([np.ones(mask.sum()), Z_ref_at_pj[mask]]).T
            y = Z_obs_at_pi[mask]
            aa, bb = np.linalg.lstsq(X, y, rcond=None)[0]
            a[k], b[k] = aa, bb
    a = pd.Series(a).interpolate().bfill().ffill().rolling(11, center=True, min_periods=3).mean().bfill().ffill().values
    b = pd.Series(b).interpolate().bfill().ffill().rolling(11, center=True, min_periods=3).mean().bfill().ffill().values
    return a, b

Z17_on21 = Z17[PJ21]; Z21_on21 = Z21_2[PI21]
Z17_on25 = Z17[PJ25]; Z25_on25 = Z25_2[PI25]
a21, b21 = local_ab(u21, Z17_on21, Z21_on21, u_ref, win=LOCAL_WIN_U)
a25, b25 = local_ab(u25, Z17_on25, Z25_on25, u_ref, win=LOCAL_WIN_U)
# 单调关系：2017 ≥ 2021 ≥ 2025（轻量约束）
b21 = np.minimum(b21, 1.0 - 0.02); b25 = np.minimum(b25, b21 - 0.02)

# ========= 在 u=1 估 2025 的 AHR_peak =========
Z_peak_25 = a25[-1] + b25[-1] * Z17[-1]
ahr_peak_25 = float(2**Z_peak_25)

# ========= 可视化 =========
y17_ref = 2**Z17
y21_proj = 2**(a21 + b21*Z17)
y25_proj = 2**(a25 + b25*Z17)

plt.figure(figsize=(12.6,4.9))
plt.plot(u_ref, y17_ref, label="2017（参考）")
plt.plot(u_ref, y21_proj, label="2021（退火后对齐）")
plt.plot(u_ref, y25_proj, label="2025（退火后对齐）")
plt.axvline(1.0, ls="--", c="gray", lw=1)
plt.scatter([1.0],[ahr_peak_25], marker="*", s=130, label=f"2025 预测峰 AHR={ahr_peak_25:.3f}")
plt.xlabel("统一进度 u（0→peak）"); plt.ylabel("AHR999")
ttl = f"三轮同轴（不混合）＋迭代动态退火｜剩余≈{days_to_peak:.0f}天 → {pred_peak_date.date()}｜VOL×{VOL_TARGET_MULT}"
plt.title(ttl); plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "ahr999_three_overlap_iter_volnorm.png", dpi=170); plt.close()

# ========= 导出指标 =========
pd.DataFrame({
    "param":["slope_tail_2025","j_end_2025","remain_tpl","days_to_peak",
             "pred_peak_date","ahr_peak_2025","VOL_TARGET_MULT"],
    "value":[slope25,int(j_end25),int(remain_tpl),float(days_to_peak),
             pred_peak_date.date(), ahr_peak_25, VOL_TARGET_MULT]
}).to_csv(OUTDIR/"ahr999_three_overlap_iter_volnorm_metrics.csv", index=False, encoding="utf-8-sig")

print("已保存：")
print("  图：", (OUTDIR/"ahr999_three_overlap_iter_volnorm.png").resolve())
print("  指标：", (OUTDIR/"ahr999_three_overlap_iter_volnorm_metrics.csv").resolve())


ValueError: operands could not be broadcast together with shapes (703,) (500,) 